# Banknote Authentication – Preprocessing Pipeline
SCT213-C002-0026/2023 - Caroline Nyanjui

SCT213-C002-0130/2023 - Catherine Njogu

**ML II Unsupervised Learning Capstone – Step 2**

## 1. Imports

In [45]:
import pandas as pd
import numpy as np

# Preprocessing
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


## 2. Load Dataset

In [39]:
data = pd.read_csv(r"C:\Users\hp\Downloads\data_banknote_authentication (1).txt")
data = pd.read_csv(r"C:\Users\hp\Downloads\data_banknote_authentication (1).txt", 
                   header=None,
                   names=['variance', 'skewness', 'curtosis', 'entropy', 'class'])

# Print first 5 rows
data.head()

,variance,skewness,curtosis,entropy,class
0,3.62160,8.6661,-2.8073,-0.44699,0
1,4.54590,8.1674,-2.4586,-1.46210,0
2,3.86600,-2.6383,1.9242,0.10645,0
3,3.45660,9.5228,-4.0112,-3.59440,0
4,0.32924,-4.4552,4.5718,-0.98880,0


## 3. Remove Duplicates

We drop duplicate rows to ensure each observation is unique before preprocessing.

## 4. Remove Target Column

Since clustering is **unsupervised**, we drop the `class` column from `X`. It is not used during preprocessing or clustering.

## 5. Outlier Flagging (IQR Method)

Outliers are **flagged but not removed** — this is intentional. Removing outliers in unsupervised learning can distort the natural structure of the data. Instead, an `outlier_flag` column is added to inform the clustering algorithms.

In [40]:
data = data.drop_duplicates()
X = data.drop('class', axis=1)

Q1 = X.quantile(0.25)
Q3 = X.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

X['outlier_flag'] = ((X < lower_bound) | (X > upper_bound)).any(axis=1)
print(X)

      variance  skewness  curtosis  entropy  outlier_flag
0      3.62160   8.66610   -2.8073 -0.44699         False
1      4.54590   8.16740   -2.4586 -1.46210         False
2      3.86600  -2.63830    1.9242  0.10645         False
3      3.45660   9.52280   -4.0112 -3.59440         False
4      0.32924  -4.45520    4.5718 -0.98880         False
...        ...       ...       ...      ...           ...
1367   0.40614   1.34920   -1.4501 -0.55949         False
1368  -1.38870  -4.87730    6.4774  0.34179         False
1369  -3.75030 -13.45860   17.5932 -2.77710          True
1370  -3.56370  -8.38270   12.3930 -1.28230          True
1371  -2.54190  -0.65804    2.6842  1.19520         False

[1348 rows x 5 columns]


## 6. Preprocessing Pipeline

We build a `sklearn` Pipeline with two steps:
- **Imputer** – fills any missing values using the median strategy
- **Scaler** – standardizes all features to have mean = 0 and standard deviation = 1

Standardization is critical for distance-based clustering algorithms like K-Means and DBSCAN.

In [46]:
numeric_features = X.columns.tolist()
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numeric_features)
])

## 7. PCA Dataset

We apply **Principal Component Analysis (PCA)** to reduce the feature space to 2 dimensions. This is useful for:
- Visualizing clusters in 2D
- Reducing noise in the data
- Speeding up clustering algorithms

Two datasets are produced:
- `df_no_pca` – all scaled features retained
- `df_with_pca` – reduced to PC1 and PC2 only

In [42]:
X_no_pca = preprocessor.fit_transform(X)

df_no_pca = pd.DataFrame(
    X_no_pca,
    columns=numeric_features
)

print(df_no_pca)



      variance  skewness  curtosis   entropy  outlier_flag
0     1.109709  1.151820 -0.975529  0.346132     -0.269062
1     1.432683  1.066810 -0.894937 -0.140707     -0.269062
2     1.195109 -0.775147  0.118015  0.611558     -0.269062
3     1.052054  1.297854 -1.253774 -1.163342     -0.269062
4    -0.040724 -1.084859  0.729928  0.086284     -0.269062
...        ...       ...       ...       ...           ...
1343 -0.013853 -0.095431 -0.661853  0.292178     -0.269062
1344 -0.641015 -1.156810  1.170350  0.724425     -0.269062
1345 -1.466217 -2.619593  3.739432 -0.771371      3.716610
1346 -1.401014 -1.754347  2.537563 -0.054476      3.716610
1347 -1.043972 -0.437589  0.293666  1.133715     -0.269062

[1348 rows x 5 columns]


In [43]:
pca = PCA(n_components=2)
X_with_pca = pca.fit_transform(X_no_pca)

df_with_pca = pd.DataFrame(
    X_with_pca,
    columns=['PC1', 'PC2']
)

print(df_with_pca)

           PC1       PC2
0    -1.674836  0.502793
1    -1.794788  0.342299
2     0.118302  1.319586
3    -2.222742 -0.624601
4     1.000248  0.512508
...        ...       ...
1343 -0.378465  0.310905
1344  1.657590  0.725631
1345  5.479560 -2.053785
1346  4.380545 -1.838778
1347  0.943101  0.573389

[1348 rows x 2 columns]


## 8. Explained Variance & Saving Datasets

We check how much information is retained after PCA.

In [44]:
explained_variance = pca.explained_variance_ratio_

variance_summary = pd.DataFrame({
    'Principal_Component': ['PC1', 'PC2'],
    'Explained_Variance_Ratio': explained_variance
})

display(variance_summary)
print("Total Variance Retained:", explained_variance.sum())

,Principal_Component,Explained_Variance_Ratio
0,PC1,0.468061
1,PC2,0.324925


Total Variance Retained: 0.7929863607416525
